In [ ]:
import matplotlib
%matplotlib inline
from eval_utils import get_text_processors, get_s2t, get_prompt, get_item, sample
from utils import set_hparams, HParams, load_checkpoint, load_a2flow, load_dp, load_vocoder

**Google Drive Link for checkpoints: https://drive.google.com/drive/folders/1i8AxolY5UED2_f4FBkdieTgdkkYxlZI0?usp=drive_link \
Download the checkpts folder and copy it into the current directory as "./checkpts". Then, run the code below.**

## Load Model Checkpoints

### Load Model Configs

In [ ]:
args = HParams()
args.generator_path = "./checkpts/a2flow-internal"
hps_generator = set_hparams(args.generator_path)

# dp for English
args.dp_path = "./checkpts/a2flow-dp-en"
hps_dp = set_hparams(args.dp_path)

vocoder_path = "./checkpts/bigvgan_v2.pt"
vocoder_config_path = "./checkpts/bigvgan_v2-config.json"

### Define Text Processors

In [ ]:
text_processors = get_text_processors(hps_generator.data.text_processors_config_path)

### Load Models (A2Flow, DP, BigVGAN)

In [ ]:
generator = load_a2flow(args.generator_path, hps_generator, n_language=len(text_processors)).cuda().eval()        

In [ ]:
dp = load_dp(args.dp_path, hps_dp, n_language=len(text_processors)).cuda().eval()

In [ ]:
vocoder = load_vocoder(vocoder_config_path, vocoder_path).cuda()

# Enter information related to the prompt and the sentences to be generated.

#### TIP: For optimal sampling, it is important to set the `num_second` of the prompt to stop at a word boundary rather than in the middle of a word. It is recommended to load a complete sentence of around 5 seconds and input its transcript for use. Additionally, the model generates better samples when the combined length of the prompt and the audio to be generated is 20 seconds or less.

`sample_path`: The sample path for the prompt \
`num_second`: The duration in seconds from the start of the prompt audio to be used \
`transcript`: The transcript for the prompt, if already known. Default=None \
`text_list`: A list containing the sentences to be generated \
`use_whisper`: Which ASR model to use for transcribing the prompt

In [ ]:
sample_path = "samples/VALLE/sample_1.wav"
num_second = 3
transcript = None
text_list = [
    'They moved thereafter cautiously about the hut, groping before and about them to find something to show that Warrenton had fulfilled his mission.',
    'And lay me down in thy cold bed, and leave my shining lot.',
    'Number ten, fresh nelly is waiting on you. -Good night husband!',
    'Yea, his honourable worship is within. But he hath a godly minister or two with him, and likewise a leech.',
    'Instead of shoes, the old man wore boots with turnover tops, and his blue coat had wide cuffs of gold braid.',
    'The army found the people in poverty and left them in comparative wealth.',
    'Thus did this humane and right-minded father comfort his unhappy daughter; and her mother, embracing her again, did all she could to soothe her feelings.',
    'He was in deep converse with the clerk, and entered the hall holding him by the arm.'
]
use_whisper = True

**Load the initial `num_second` seconds of the prompt audio from the beginning.** 

**If `transcript = None`, use an ASR model (such as Whisper or HuBERT-L) to transcribe it.**

In [ ]:
# Load Prompt
wav, mel, transcript = get_prompt(sample_path, num_second, transcript, use_whisper)

**Since this ASR transcription may be inaccurate, it’s recommended to listen to the Prompt Audio above and correct the transcript as needed.**

In [ ]:
# if you want to correct the ASR transcript, edit the transcript below. 
# transcript = "TYPE HERE"

# Sampling

`args.language`: The language in which to generate the content \
`args.timesteps`: The number of sampling steps for generation; 32 or higher is recommended, but larger values slow down sampling. (Default) 32 \
`args.gradient_scale`: The classifier-free guidance scale. (Default) 2 \
`args.alpha`: The time-shifting scale. (Default) 3 \
`args.fp16`: False \
`args.dur_scale`: A variable for scaling the total length proportionally to `dur_scale`. (Default) 1.  (min: 0.8, max:1.2)

In [ ]:
args.language = 'en_US'
args.timesteps = 32
args.gradient_scale = 2.
args.alpha = 3.
args.fp16 = False
args.dur_scale = 1.
args.use_prompt = True
args.use_dp = False

sample(args, get_item(text_list, wav, mel, transcript), text_processors, generator, dp, vocoder) 